# 02 — Neural Activation Analysis

Mechanistic interpretability: value functions, activation contrasts, linear probes,
activation patching, and representational similarity analysis.

**Prerequisites:** Run `python interpretability/run_collection.py --model-path artifacts/good_old/ckpt_best.pt` first.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.decomposition import PCA

from cogniland.env.types import EnvConfig
from cogniland.env.core import compute_minimap_batch, compute_terrain_levels
from cogniland.models.ppo import ActorCritic
from cogniland.utils import load_checkpoint
from interpretability.data_manager import TrajectoryDataManager
from interpretability import probes, viz

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'figure.dpi': 120})
sns.set_style('whitegrid')

In [ ]:
# Load model — auto-detect architecture from checkpoint
import math
device = 'cpu'
CKPT_PATH = '../../artifacts/good_old/ckpt_best.pt'

_ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
_sd = _ckpt['model_state_dict']
_arch = dict(
    scalar_dim=_sd['scalar_net.0.weight'].shape[1],
    minimap_channels=_sd['cnn.0.weight'].shape[1],
    hidden_dim=_sd['trunk.0.weight'].shape[0],
    action_dim=_sd['actor.weight'].shape[0],
    cnn_channels=_sd['cnn.3.weight'].shape[0],
    cnn_out_spatial=int(math.isqrt((_sd['trunk.0.weight'].shape[1] - _sd['scalar_net.0.weight'].shape[0]) // _sd['cnn.3.weight'].shape[0])),
    scalar_hidden=_sd['scalar_net.0.weight'].shape[0],
)
print(f'Architecture: {_arch}')

model = ActorCritic(**_arch).to(device)
load_checkpoint(CKPT_PATH, model, device=device)
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# Load data
dm = TrajectoryDataManager('../../interpretability/data/')
env_config = EnvConfig(device=device)
compiled = env_config.compile_terrain(device)
print(f'{dm.n_trajectories} trajectories loaded')

## 1. Value Function Visualization

In [ ]:
# Pick a successful trajectory for value function overlay
successful = dm.filter(outcome='success', map_source='test')
if len(successful) == 0:
    successful = dm.filter(map_source='test')

traj_row = successful.iloc[0]
traj_id = int(traj_row['traj_id'])
tdata = dm.get_trajectory(traj_id)
spawn = (int(tdata['attrs']['spawn_row']), int(tdata['attrs']['spawn_col']))
target = (int(tdata['attrs']['target_row']), int(tdata['attrs']['target_col']))
map_id = int(traj_row['map_id'])

# Load the map
test_data = torch.load('../../data/test_seed42_n16.pt', map_location='cpu', weights_only=True)
world_map = test_data['maps'][map_id]
wm_np = world_map.numpy()
print(f'Trajectory {traj_id}: map {map_id}, {traj_row["outcome"]}, {traj_row["episode_length"]} steps')

In [ ]:
# Compute V(s) across land positions
land_thresh = compiled.land_threshold
stride = 5
land_positions = []
for r in range(0, 250, stride):
    for c in range(0, 250, stride):
        if wm_np[r, c] > land_thresh:
            land_positions.append([r, c])

positions = torch.tensor(land_positions, dtype=torch.long)
B = positions.shape[0]
target_t = torch.tensor([list(target)], dtype=torch.long).expand(B, 2)

values = []
batch_size = 256
for start in range(0, B, batch_size):
    end = min(start + batch_size, B)
    pos_b = positions[start:end]
    b = pos_b.shape[0]
    wm_b = world_map.unsqueeze(0).expand(b, -1, -1)
    tgt_b = target_t[start:end]

    terrain_idx = compute_terrain_levels(wm_b, pos_b, compiled)
    minimap = compute_minimap_batch(
        wm_b, pos_b, env_config.minimap_max_ray,
        terrain_idx, env_config.minimap_occlude,
        env_config.minimap_clear_tolerance, compiled,
        target_pos=tgt_b,
    )
    compass = (tgt_b - pos_b).float()
    compass = compass / compass.norm(dim=1, keepdim=True).clamp(min=1e-8)
    scalars = torch.stack([
        compass[:, 0], compass[:, 1],
        terrain_idx / max(compiled.num_terrains - 1, 1),
        torch.ones(b) * 0.5, torch.ones(b) * 1.0,
    ], dim=1)
    obs = {'scalars': scalars, 'minimap': minimap}
    with torch.no_grad():
        v = model.get_value(obs).numpy()
    values.append(v)

value_arr = np.concatenate(values)
print(f'Computed V(s) for {len(value_arr)} positions')
print(f'V range: [{value_arr.min():.2f}, {value_arr.max():.2f}]')

In [ ]:
fig = viz.plot_value_function_overlay(
    wm_np, value_arr, positions.numpy(), compiled,
    trajectory=tdata['positions'], target=target, spawn=spawn,
    title=f'Value Function — Map {map_id}',
)
plt.show()

## 2. Activation Trajectories

In [ ]:
# PCA of trunk.2 activations over one trajectory
trunk_acts = tdata['activations'].get('trunk_2')
if trunk_acts is not None:
    acts_f32 = trunk_acts.astype(np.float32)
    if acts_f32.ndim > 2:
        acts_f32 = acts_f32.reshape(acts_f32.shape[0], -1)
    pca_traj = PCA(n_components=3)
    pcs = pca_traj.fit_transform(acts_f32)
    terrain = np.array(tdata['terrain_idx'])

    events = {}
    flags = tdata.get('flags', {})
    if 'target_just_entered_view' in flags:
        events['target_enters_view'] = list(np.where(flags['target_just_entered_view'])[0])
    if 'is_on_water' in flags:
        w = flags['is_on_water']
        events['water_entry'] = [i for i in range(1, len(w)) if w[i] and not w[i-1]]
    if 'is_low_hp' in flags:
        events['low_hp'] = list(np.where(flags['is_low_hp'])[0][:5])

    fig = viz.plot_activation_over_trajectory(
        pcs, terrain, compiled, events=events,
        pc_labels=[f'PC{i+1} ({pca_traj.explained_variance_ratio_[i]:.1%})' for i in range(3)],
    )
    plt.show()
else:
    print('No trunk_2 activations available')

In [ ]:
# Global activation PCA colored by terrain
all_acts, all_tids, all_terrain = dm.get_all_activations_flat('trunk_2')
print(f'Total steps: {len(all_acts)}')

# Subsample if needed
if len(all_acts) > 50000:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(all_acts), 50000, replace=False)
    all_acts = all_acts[idx]
    all_terrain = all_terrain[idx]

pca_global = PCA(n_components=2)
emb_global = pca_global.fit_transform(all_acts)

terrain_colors = viz.get_terrain_colors(compiled)
colors = np.array([terrain_colors[int(t)] if 0 <= int(t) < 9 else [0.5, 0.5, 0.5]
                   for t in all_terrain])

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(emb_global[:, 0], emb_global[:, 1], c=colors, s=3, alpha=0.3, edgecolors='none')
for i in range(9):
    ax.scatter([], [], c=[terrain_colors[i]], s=40, label=viz.TERRAIN_NAMES[i])
ax.legend(fontsize=9, markerscale=2, loc='upper right')
ax.set_xlabel(f'PC1 ({pca_global.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_global.explained_variance_ratio_[1]:.1%})')
ax.set_title('Trunk.2 Activations Colored by Terrain')
plt.show()

## 3. Conditional Activation Contrasts

In [ ]:
# Contrast 1: HP/resources
_, _, delta_hp = probes.compute_activation_contrast(
    dm,
    positive_flags={'is_low_hp': True, 'is_low_resources': True},
    negative_flags={'is_low_hp': False, 'is_low_resources': False},
    layer='trunk_2',
)
if len(delta_hp) > 1:
    fig = viz.plot_activation_bar(delta_hp, title='Low vs High HP+Resources (trunk.2)')
    plt.show()
else:
    print('Not enough data for HP contrast')

In [ ]:
# Contrast 2: Target visible
_, _, delta_tgt = probes.compute_activation_contrast(
    dm,
    positive_flags={'is_target_visible': True},
    negative_flags={'is_target_visible': False},
    layer='trunk_2',
)
if len(delta_tgt) > 1:
    fig = viz.plot_activation_bar(delta_tgt, title='Target Visible vs Not (trunk.2)')
    plt.show()

In [ ]:
# Contrast 3: Forest
_, _, delta_forest = probes.compute_activation_contrast(
    dm,
    positive_flags={'is_in_forest': True},
    negative_flags={'is_in_forest': False},
    layer='trunk_2',
)
if len(delta_forest) > 1:
    fig = viz.plot_activation_bar(delta_forest, title='In Forest vs Not (trunk.2)')
    plt.show()

## 4. Linear Probes

In [ ]:
layers = ['trunk_0', 'trunk_2', 'actor', 'critic']
all_probe_results = {}

# Probe: Low HP
all_probe_results['Low HP'] = probes.run_probes_for_concept(
    dm, 'low_hp',
    positive_flags={'is_low_hp': True},
    negative_flags={'is_low_hp': False},
    layers=layers,
)

# Probe: Target visible
all_probe_results['Target Visible'] = probes.run_probes_for_concept(
    dm, 'target_visible',
    positive_flags={'is_target_visible': True},
    negative_flags={'is_target_visible': False},
    layers=layers,
)

# Probe: Forest
all_probe_results['Forest'] = probes.run_probes_for_concept(
    dm, 'in_forest',
    positive_flags={'is_in_forest': True},
    negative_flags={'is_in_forest': False},
    layers=layers,
)

# Probe: Water
all_probe_results['Water'] = probes.run_probes_for_concept(
    dm, 'on_water',
    positive_flags={'is_on_water': True},
    negative_flags={'is_on_water': False},
    layers=layers,
)

# Print results
for concept, layer_results in all_probe_results.items():
    print(f'\n{concept}:')
    for layer, res in layer_results.items():
        print(f'  {layer}: train={res.get("train_acc", 0):.3f}, test={res.get("test_acc", 0):.3f}')

In [ ]:
fig = viz.plot_linear_probe_accuracy(all_probe_results)
plt.show()

## 5. Activation Patching (Causal Intervention)

In [ ]:
# Get steps with target visible and not visible
vis_steps = dm.get_steps_where(is_target_visible=True)
novis_steps = dm.get_steps_where(is_target_visible=False)
print(f'Target visible steps: {len(vis_steps["traj_ids"])}')
print(f'Target not visible steps: {len(novis_steps["traj_ids"])}')

In [ ]:
# Build obs from a visible and not-visible step
if len(vis_steps['traj_ids']) > 0 and len(novis_steps['traj_ids']) > 0:
    vis_tid = int(vis_steps['traj_ids'][0])
    vis_sidx = int(vis_steps['step_indices'][0])
    novis_tid = int(novis_steps['traj_ids'][0])
    novis_sidx = int(novis_steps['step_indices'][0])

    vis_traj = dm.get_trajectory(vis_tid)
    novis_traj = dm.get_trajectory(novis_tid)

    def _obs(tdata, step):
        s = torch.tensor(tdata['obs_scalars'][step], dtype=torch.float32).unsqueeze(0)
        if 'obs_minimaps' in tdata and len(tdata['obs_minimaps']) > step:
            m = torch.tensor(tdata['obs_minimaps'][step], dtype=torch.float32).unsqueeze(0)
        else:
            m = torch.zeros(1, 3, 45, 45)
        return {'scalars': s, 'minimap': m}

    obs_vis = _obs(vis_traj, vis_sidx)
    obs_novis = _obs(novis_traj, novis_sidx)

    # Full-layer patching
    result = probes.activation_patching(model, obs_novis, obs_vis, patch_layer='trunk.2')
    print(f'Full trunk.2 patch KL divergence: {result["kl_divergence"]:.6f}')
    print(f'Baseline action probs: {result["baseline_probs"]}')
    print(f'Patched action probs:  {result["patched_probs"]}')

    # Per-neuron patching
    top_neurons, kl_vals = probes.batch_activation_patching_by_neuron(
        model, obs_novis, obs_vis, patch_layer='trunk.2', top_k=20,
    )
    fig = viz.plot_activation_patching_results(
        top_neurons, kl_vals,
        title='Activation Patching: Target Visibility (trunk.2, per neuron)',
    )
    plt.show()
else:
    print('Not enough data for patching analysis')

## 6. Representational Similarity Analysis (RSA)

In [ ]:
rdm_results = probes.compute_terrain_rdms(
    dm, layers=['cnn_0', 'cnn_5', 'trunk_0', 'trunk_2'],
)

if rdm_results:
    n = len(rdm_results)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6))
    if n == 1:
        axes = [axes]
    for i, (layer, (rdm, names)) in enumerate(rdm_results.items()):
        viz.plot_rdm(rdm, names, title=f'RDM — {layer}', ax=axes[i])
    fig.tight_layout()
    plt.show()
else:
    print('No RDM data available')

## 7. Strategy-Specific Activations (LDA)

In [ ]:
# Load cluster labels from notebook 01 (or recompute)
if 'cluster' not in dm.summary.columns:
    print('Run notebook 01 first to compute clusters.')
    print('Recomputing...')
    from interpretability.trajectory_features import featurize_all
    import umap, hdbscan
    features, _ = featurize_all(dm.summary, str(dm.h5_path))
    pca = PCA(n_components=min(features.shape[1], features.shape[0]) - 1)
    pca.fit(features)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_keep = max(int(np.searchsorted(cumvar, 0.95)) + 1, 2)
    features_pca = pca.transform(features)[:, :n_keep]
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    emb = reducer.fit_transform(features_pca)
    labels = hdbscan.HDBSCAN(min_cluster_size=max(3, len(features)//10), min_samples=2).fit_predict(emb)
    dm.summary['cluster'] = labels

# Per-trajectory mean trunk.2 activations
traj_means = []
traj_labels = []
for _, row in dm.summary.iterrows():
    tid = int(row['traj_id'])
    lab = int(row.get('cluster', -1))
    acts_list = dm.get_activations([tid], layer='trunk_2')
    if acts_list and len(acts_list[0]) > 0:
        a = acts_list[0].astype(np.float32)
        if a.ndim > 2:
            a = a.reshape(a.shape[0], -1)
        traj_means.append(a.mean(axis=0))
        traj_labels.append(lab)

if traj_means:
    traj_means = np.stack(traj_means)
    traj_labels = np.array(traj_labels)

    projected, lda = probes.cluster_lda(traj_means, traj_labels, n_components=2)

    fig, ax = plt.subplots(figsize=(10, 8))
    for lab in sorted(set(traj_labels)):
        mask = traj_labels == lab
        label = f'Cluster {lab}' if lab >= 0 else 'Noise'
        ax.scatter(projected[mask, 0], projected[mask, 1], s=30, alpha=0.7, label=label)
    ax.legend()
    ax.set_xlabel('LDA 1')
    ax.set_ylabel('LDA 2')
    ax.set_title('LDA Projection: Strategy Clusters in Activation Space')
    plt.show()